# CAMEL Data Gen Cookbook


You can also check this cookbook in colab [here](https://colab.research.google.com/drive/1MHnHcDP-Gg2BKxP8EVPivmEeOX7QtxWE?usp=sharing)

## Overview

This tutorial guides you through generating user queries and structured tool call data using ChatAgent and various toolkits. We'll cover setting up the model, generating user-like queries, and saving the output in JSON format.


### Installation

Ensure you have CAMEL AI installed in your Python environment:

In [18]:
!pip install "camel-ai[all]==0.2.6"

   ---------------------------------------- 0.0/394.3 kB ? eta -:--:--
   ------------------------------------ --- 358.4/394.3 kB 7.4 MB/s eta 0:00:01
   ---------------------------------------- 394.3/394.3 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: camel-ai
    Found existing installation: camel-ai 0.2.9
    Uninstalling camel-ai-0.2.9:
      Successfully uninstalled camel-ai-0.2.9



[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1: Import Required Libraries and Modules
Start by importing necessary libraries and modules.

In [19]:
import uuid
import json
from typing import Callable, Dict, List, Union

# Import necessary classes and functions from camel library
from camel.agents import ChatAgent
from camel.models import ModelFactory
from camel.toolkits import FunctionTool, SearchToolkit, MathToolkit, ArxivToolkit, GoogleMapsToolkit, RetrievalToolkit, WeatherToolkit
from camel.types import ModelPlatformType, ModelType

In [20]:
import os
# from getpass import getpass
# Prompt for the OpenAI API key securely
# openai_api_key = getpass('Enter your API key: ')
os.environ["OPENAI_API_KEY"] = ""

# google_api_key = getpass('Enter your API key: ')
# os.environ["GOOGLE_API_KEY"] = google_api_key

# weather_api_key = getpass('Enter your API key: ')
# os.environ["OPENWEATHERMAP_API_KEY"] = weather_api_key

## Step 2: Define Function to Generate User Queries
This function leverages specific tools to generate human-like queries. We’ll set up a ChatAgent to create relevant, tool-specific user queries.

In [21]:
def generate_user_query(selected_tool: Union[Callable, FunctionTool], n: int = 1) -> List[str]:
    r"""Generates user queries by leveraging specific tools, helping the ChatAgent craft
    human-like queries that take advantage of each tool's functionality.
    """
    # Define system message to guide the ChatAgent on how to use the tools effectively
    tool_call_sys_msg = (
        "You are a smart query generator designed to utilize specific tools based on user needs. "
        "Formulate queries as a human user would, focusing on how each tool can help achieve the goal.\n\n"

        "Instructions:\n"
        "1. Craft a precise and actionable query that leverages the provided tools.\n"
        "2. Ensure the query is relevant to the user's intent and aligned with each tool's capabilities.\n"
        "3. Use only the tools available, adapting the query to match their specific functions.\n"
    )

    # Convert to FunctionTool if necessary
    if not isinstance(selected_tool, FunctionTool):
        selected_tool = FunctionTool(selected_tool)

    # Create a model instance for generating queries
    query_model = model = ModelFactory.create(
    model_platform=ModelPlatformType.OPENAI_COMPATIBLE_MODEL,
    model_type="deepseek-chat",
    api_key=os.environ.get("OPENAI_COMPATIBILIY_API_KEY"),
    url=os.environ.get("OPENAI_COMPATIBILIY_API_BASE_URL"),
    model_config_dict={"temperature": 0.4, "max_tokens": 4096},
)


    # Initialize ChatAgent with the system message
    query_agent = ChatAgent(
        system_message=tool_call_sys_msg,
        model=query_model,
    )
    queries = []

    query_agent.reset()  # Reset for a fresh start

    # Prepare tools info message for guiding query generation
    tools_info_message = (
        "Generate a relevant query based on the following tool details:\n\n" +
        "\n".join(f"Tool Schema: {selected_tool.get_openai_tool_schema()}")
    )

    # Generate queries
    for _ in range(n):
        response = query_agent.step(tools_info_message)
        queries.append(response.msgs[0].content)  # Extract the generated query

    return queries


## Step 3: Define Function to Generate Structured Tool Call Data
This function will structure tool call data based on user queries by leveraging each selected tool.

In [22]:
def generate_tool_call_data(user_messages: List[str], selected_tool: Union[Callable, FunctionTool]) -> Dict[str, List[Dict]]:
    r"""Generates structured tool call data for a list of user messages by using each specified tool in selected_tools.
    """

    # Convert to FunctionTool if necessary
    if not isinstance(selected_tool, FunctionTool):
        selected_tool = FunctionTool(selected_tool)

    # Define system message guiding ChatAgent on function calls
    tool_call_sys_msg = (
        "You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. "
        "You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug into functions."
    )

    # Initialize model for tool call data generation
    tool_model = model = ModelFactory.create(
    model_platform=ModelPlatformType.OPENAI_COMPATIBLE_MODEL,
    model_type="deepseek-chat",
    api_key=os.environ.get("OPENAI_COMPATIBILIY_API_KEY"),
    url=os.environ.get("OPENAI_COMPATIBILIY_API_BASE_URL"),
    model_config_dict={"temperature": 0.4, "max_tokens": 4096},
)


    all_tool_data = {}

    tool_schema = selected_tool.get_openai_tool_schema()

    # Set up ChatAgent with the system message, model, and the single tool
    tool_agent = ChatAgent(
        system_message=tool_call_sys_msg,
        model=tool_model,
        external_tools=[selected_tool],
    )

    tool_agent.reset()

    tool_output_data = []
    for user_message in user_messages:

        # Generate response using ChatAgent and structure output
        response = tool_agent.step(user_message)

        if response.info["external_tool_request"]:
            tool_output_data.append([
                {"id": str(uuid.uuid4())},
                {
                    "from": "system",
                    "value": tool_call_sys_msg,
                    "tool_schema": tool_schema,
                },
                {"from": "human", "value": user_message},
                {
                    "from": "gpt",
                    "value": [
                        {"arguments": response.info["external_tool_request"].function.arguments, "name": response.info["external_tool_request"].function.name}
                    ],
                },
            ])

    # Add generated data to the dictionary with tool name as the key
    all_tool_data[selected_tool.func.__name__] = tool_output_data

    return tool_output_data


## Step 4: Initialize Toolkits and Define Tools List
We’ll set up toolkits from different domains like math, search, and maps to expand the query generation.

In [23]:
selected_tools = [
    *MathToolkit().get_tools(),
    
    *ArxivToolkit().get_tools(),
    
    *RetrievalToolkit().get_tools(),
    
]  # Add additional tools as needed


## Step 5: Generate Data and Save to JSON
We now loop through each tool, generate queries, create tool call data, and save it all in JSON format.

In [24]:
results = {
    "generated_queries": [],
    "tool_call_data": []
}

for selected_tool in selected_tools:
    user_queries = generate_user_query(selected_tool=selected_tool, n=5)
    tool_call_data = generate_tool_call_data(user_queries, selected_tool)

    # Append results to the lists instead of overwriting
    results["generated_queries"].extend(user_queries)
    results["tool_call_data"].extend(tool_call_data)

# Specify output file path
output_file = "generated_tool_call_data.json"

# Save data to JSON file
with open(output_file, "w") as f:
    json.dump(results, f, indent=4)

print(f"Data saved to {output_file}")


NotFoundError: Error code: 404 - {'error': {'message': 'The model `deepseek-chat` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'param': None, 'code': 'model_not_found'}}

## Step 6: Verify the JSON Output
Open the generated JSON file and verify that the tool call data has been saved correctly.


[ { "from": "system", "value": "You are a function calling AI model. You are provided with function signatures within <tools> </tools> XML tags. You may call one or more functions to assist with the user query. Don't make assumptions about what values to plug into functions.\n<tools>\n[{'type': 'function', 'function': {'name': 'download_files_sequentially', 'description': 'Downloads files in a sequence from a base URL, respecting the specified crawl-delay.', 'parameters': {'type': 'object', 'properties': {'base_url': {'type': 'string', 'description': 'The base URL for file downloads, without the file number or extension.', 'enum': [None]}, 'start_episode': {'type': 'integer', 'description': 'The starting episode number for the download sequence.', 'enum': [None]}, 'end_episode': {'type': 'integer', 'description': 'The ending episode number for the download sequence.', 'enum': [None]}, 'file_extension': {'type': 'string', 'description': 'The file extension for the files to be downloaded.', 'enum': [None]}, 'crawl_delay': {'type': 'integer', 'description': 'The delay in seconds between each file download request.', 'enum': [None]}}, 'required': ['base_url', 'start_episode', 'end_episode', 'file_extension', 'crawl_delay']}}}, {'type': 'function', 'function': {'name': 'check_robots_txt', 'description': 'Checks the robots.txt file of a website to determine the crawl-delay and allowed paths for web scraping.', 'parameters': {'type': 'object', 'properties': {'website_url': {'type': 'string', 'description': 'The URL of the website to check the robots.txt file.', 'enum': [None]}}, 'required': ['website_url']}}}]\n</tools>\nFor each function call return a json object with function name and arguments within <tool_call> </tool_call> tags with the following schema:\n<tool_call>\n{'arguments': <args-dict>, 'name': <function-name>}\n</tool_call>\n" }, { "from": "human", "value": "I am currently working on a project where I need to download a large set of podcast transcripts for analysis. These transcripts are available on the website located at `http://example.com/podcasts/transcripts/` and are named sequentially, starting from 'episode1.txt' all the way up to 'episode600.txt'. To ensure that I comply with the website's terms of service and do not overload their server, I would like to implement a respectful crawl-delay between each download request, as specified by their robots.txt file. The transcripts are all in plain text (.txt) format. Could you please initiate the downloading process for the episodes 1 through 600 sequentially, with an appropriate delay between each download as indicated by the website's crawling policy? \n\nPlease use the following details for the function call:\n- Base URL: `http://example.com/podcasts/transcripts/episode`\n- Start Episode: 1\n- End Episode: 600\n- File Extension: 'txt'\n- Crawl Delay: 10 seconds (as determined by the website's robots.txt) \n\nEnsure that the function adheres to the crawl-delay policy and complete the download process for me. Thank you." }, { "from": "gpt", "value": "<tool_call>\n{'arguments': {'website_url': 'http://example.com'}, 'name': 'check_robots_txt'}\n</tool_call>\n" } ]

In [ ]:
# Load and display data to ensure correctness
with open(output_file, "r") as f:
    data = json.load(f)

print("Sample data:", json.dumps(data["generated_queries"][:100], indent=4))  # Display sample queries
print("\nSample tool call data:", json.dumps(data["tool_call_data"][:100], indent=4))  # Display sample tool call data


## Summary
In this tutorial, you learned how to generate user queries and structure tool call data by leveraging multiple tools with the camel library. This setup enables scalable and flexible data generation for various toolkits and tasks.